## Loading dataset

In [23]:
import pandas as pd

orders = pd.read_csv("/content/orders.csv")

print(orders.head())

   order_id  user_id  restaurant_id  order_date  total_amount  \
0         1     2508            450  18-02-2023        842.97   
1         2     2693            309  18-01-2023        546.68   
2         3     2084            107  15-07-2023        163.93   
3         4      319            224  04-10-2023       1155.97   
4         5     1064            293  25-12-2023       1321.91   

                  restaurant_name  
0               New Foods Chinese  
1  Ruchi Curry House Multicuisine  
2           Spice Kitchen Punjabi  
3          Darbar Kitchen Non-Veg  
4       Royal Eatery South Indian  


In [24]:
users = pd.read_json("/content/users.json")

print(users.head())

   user_id    name       city membership
0        1  User_1    Chennai    Regular
1        2  User_2       Pune       Gold
2        3  User_3  Bangalore       Gold
3        4  User_4  Bangalore    Regular
4        5  User_5       Pune       Gold


In [25]:
import sqlite3
import pandas as pd

# Create SQLite in-memory database
conn = sqlite3.connect(":memory:")

# Read SQL file
with open("/content/restaurants.sql", "r") as f:
    sql_script = f.read()

# Execute script
conn.executescript(sql_script)

# Load restaurants table into pandas
restaurants = pd.read_sql("SELECT * FROM restaurants", conn)

print(restaurants.head())

   restaurant_id restaurant_name  cuisine  rating
0              1    Restaurant_1  Chinese     4.8
1              2    Restaurant_2   Indian     4.1
2              3    Restaurant_3  Mexican     4.3
3              4    Restaurant_4  Chinese     4.1
4              5    Restaurant_5  Chinese     4.8


In [26]:
merged1 = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)


In [27]:
final_df = pd.merge(
    merged1,
    restaurants,
    on="restaurant_id",
    how="left"
)


In [28]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)

print("Final dataset created successfully!")



Final dataset created successfully!


## **SQL Quries**


In [29]:
final_df.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [30]:
gold_members = final_df[final_df['membership'] == 'Gold']
revenue_by_city = gold_members.groupby('city')['total_amount'].sum().reset_index()
revenue_by_city = revenue_by_city.rename(columns={'total_amount': 'total_revenue'})
top_city = revenue_by_city.sort_values(by='total_revenue', ascending=False).head(1)

print(top_city)

      city  total_revenue
1  Chennai     1080909.79


In [31]:
avg_order_value_by_cuisine = final_df.groupby('cuisine')['total_amount'].mean().reset_index()
avg_order_value_by_cuisine = avg_order_value_by_cuisine.rename(columns={'total_amount': 'avg_order_value'})
top_cuisine = avg_order_value_by_cuisine.sort_values(by='avg_order_value', ascending=False).head(1)

print(top_cuisine)

   cuisine  avg_order_value
3  Mexican       808.021344


In [32]:
user_total_spent = final_df.groupby('user_id')['total_amount'].sum()
high_value_users_count = user_total_spent[user_total_spent > 1000].count()

print(f"Number of high value users: {high_value_users_count}")

Number of high value users: 2544


In [33]:
# Define the rating ranges
def get_rating_range(rating):
    if 3.0 <= rating <= 3.5:
        return '3.0 - 3.5'
    elif 3.6 <= rating <= 4.0:
        return '3.6 - 4.0'
    elif 4.1 <= rating <= 4.5:
        return '4.1 - 4.5'
    elif 4.6 <= rating <= 5.0:
        return '4.6 - 5.0'
    return 'Other' # Handle cases outside defined ranges if any

# Apply the function to create a new 'rating_range' column
final_df['rating_range'] = final_df['rating'].apply(get_rating_range)

# Group by rating_range and sum total_amount to get total_revenue
revenue_by_rating_range = final_df.groupby('rating_range')['total_amount'].sum().reset_index()
revenue_by_rating_range = revenue_by_rating_range.rename(columns={'total_amount': 'total_revenue'})

# Get the rating range with the highest total revenue
top_rating_range_revenue = revenue_by_rating_range.sort_values(by='total_revenue', ascending=False).head(1)

print(top_rating_range_revenue)

  rating_range  total_revenue
3    4.6 - 5.0     2197030.75


In [34]:
gold_members_avg_order = final_df[final_df['membership'] == 'Gold']
avg_order_value_by_city = gold_members_avg_order.groupby('city')['total_amount'].mean().reset_index()
avg_order_value_by_city = avg_order_value_by_city.rename(columns={'total_amount': 'avg_order_value'})
top_city_avg_order = avg_order_value_by_city.sort_values(by='avg_order_value', ascending=False).head(1)

print(top_city_avg_order)

      city  avg_order_value
1  Chennai        808.45908


In [35]:
cuisine_summary = final_df.groupby('cuisine').agg(
    restaurant_count=('restaurant_id', 'nunique'),
    total_revenue=('total_amount', 'sum')
).reset_index()

# Sort by restaurant_count ascending, then by total_revenue descending, and get the top one
top_cuisine_by_metrics = cuisine_summary.sort_values(
    by=['restaurant_count', 'total_revenue'],
    ascending=[True, False]
).head(1)

print(top_cuisine_by_metrics)

   cuisine  restaurant_count  total_revenue
0  Chinese               120     1930504.65


In [36]:
gold_member_orders_count = final_df[final_df['membership'] == 'Gold'].shape[0]
total_orders_count = final_df.shape[0]
gold_order_percentage = (gold_member_orders_count / total_orders_count) * 100

print(f"Percentage of orders from Gold members: {gold_order_percentage:.2f}%")

Percentage of orders from Gold members: 49.87%


In [37]:
restaurant_summary = final_df.groupby('restaurant_name_y').agg(
    avg_order_value=('total_amount', 'mean'),
    total_orders=('order_id', 'count')
).reset_index()

# Filter for restaurants with less than 20 orders
filtered_restaurants = restaurant_summary[restaurant_summary['total_orders'] < 20]

# Get the restaurant with the highest average order value from the filtered list
top_restaurant_by_avg_order = filtered_restaurants.sort_values(by='avg_order_value', ascending=False).head(1)

print(top_restaurant_by_avg_order)

    restaurant_name_y  avg_order_value  total_orders
216    Restaurant_294      1040.222308            13


In [38]:
distinct_restaurant_names = final_df['restaurant_name_y'].drop_duplicates().head(20)
print(distinct_restaurant_names)

0     Restaurant_450
1     Restaurant_309
2     Restaurant_107
3     Restaurant_224
4     Restaurant_293
5     Restaurant_499
6      Restaurant_35
7      Restaurant_57
8       Restaurant_7
9     Restaurant_183
10    Restaurant_235
11    Restaurant_423
12    Restaurant_244
13    Restaurant_112
14    Restaurant_383
15    Restaurant_149
16    Restaurant_421
17    Restaurant_414
18     Restaurant_73
19     Restaurant_52
Name: restaurant_name_y, dtype: object


In [39]:
df_filtered = final_df[
    ((final_df['membership'] == 'Gold') & (final_df['cuisine'] == 'Indian')) |
    ((final_df['membership'] == 'Gold') & (final_df['cuisine'] == 'Italian')) |
    ((final_df['membership'] == 'Regular') & (final_df['cuisine'] == 'Indian')) |
    ((final_df['membership'] == 'Regular') & (final_df['cuisine'] == 'Chinese'))
]

grouped_df = df_filtered.groupby(['membership', 'cuisine'])['total_amount'].sum().reset_index()
grouped_df = grouped_df.rename(columns={'total_amount': 'total_revenue'})

grouped_df['combination'] = grouped_df['membership'] + ' + ' + grouped_df['cuisine']

result = grouped_df.sort_values(by='total_revenue', ascending=False)

print(result[['combination', 'total_revenue']])

         combination  total_revenue
1     Gold + Italian     1005779.05
3   Regular + Indian      992100.27
0      Gold + Indian      979312.31
2  Regular + Chinese      952790.91


In [40]:
final_df['order_date'] = pd.to_datetime(final_df['order_date'], format='%d-%m-%Y')
final_df['quarter'] = final_df['order_date'].dt.quarter.apply(lambda x: f'Q{x}')

revenue_by_quarter = final_df.groupby('quarter')['total_amount'].sum().reset_index()
top_quarter = revenue_by_quarter.sort_values(by='total_amount', ascending=False).head(1)

print(top_quarter)

  quarter  total_amount
2      Q3     2037385.1


In [41]:
total_gold_orders = final_df[final_df['membership'] == 'Gold'].shape[0]

print(f"Total orders from Gold members: {total_gold_orders}")

Total orders from Gold members: 4987


In [43]:
total_revenue_hyderabad = final_df[final_df['city'] == 'Hyderabad']['total_amount'].sum()
print(f"Total revenue for Hyderabad: {round(total_revenue_hyderabad)}")

Total revenue for Hyderabad: 1889367


In [45]:
distinct_user_count = final_df['user_id'].nunique()

print(f"Number of distinct users: {distinct_user_count}")

Number of distinct users: 2883


In [47]:
gold_members = final_df[final_df['membership'] == 'Gold']
avg_order_value = gold_members['total_amount'].mean()

print(f"Average order value for Gold members: {avg_order_value:.2f}")

Average order value for Gold members: 797.15


In [49]:
highly_rated_orders = final_df[final_df['rating'] >= 4.5]
total_highly_rated_orders = highly_rated_orders.shape[0]

print(f"Total orders from restaurants with rating >= 4.5: {total_highly_rated_orders}")

Total orders from restaurants with rating >= 4.5: 3374


In [51]:
gold_members = final_df[final_df['membership'] == 'Gold']
revenue_by_city = gold_members.groupby('city')['total_amount'].sum()
top_city_name = revenue_by_city.idxmax()

total_orders_top_gold_city = gold_members[gold_members['city'] == top_city_name].shape[0]

print(f"Total orders from Gold members in the top city ({top_city_name}): {total_orders_top_gold_city}")

Total orders from Gold members in the top city (Chennai): 1337


In [53]:
total_rows = final_df.shape[0]

print(f"Total rows in final_food_delivery_dataset: {total_rows}")

Total rows in final_food_delivery_dataset: 10000
